# 🧹 Meningioma Cleaning Notebook

Builds the analysis-ready cohort and modelling datasets.

Run top to bottom. Hands off to `meningioma-modelling.ipynb` via `output/datasets/`.


Run top to bottom. Creates `output/datasets/unimputed_df.parquet` plus a MICE or simple modelling parquet; §16 validates every saved parquet before handoff.

<details>
<summary><b>Pipeline map</b> — notebook step → <code>config/*.py</code></summary>

Each step calls a numbered config module (the pipeline engine) or a shared helper. Study-specific choices live in the notebook cells; the `config/` files are just the machinery.

| Step | What it does | Driven by |
|------|--------------|-----------|
| 00 | Setup — imports, reset `output/` | — |
| 01 | Load raw export | `config/01_cohort.py · load_raw` |
| 02 | Rename columns → snake_case | `config/02_column_rename_map.py` |
| 03 | Cohort filter (optional year subset) | `config/01_cohort.py · filter_cohort` |
| 04 | Infer + override schema | `config/03_schema_overrides.py` |
| 05 | Row filters — pre-schema (raw strings) | `config/04_row_filters.py` |
| 06 | Apply schema (coerce types) | `cleaning.apply_schema` |
| 07 | Duplicate audit | `cleaning.audit_duplicates` |
| 08 | Row filters — post-schema | `config/04_row_filters.py` |
| 09 | DDA — first descriptive pass | `dda.run_dda` |
| 10 | Missingness story (diagnostic) | `analyze_missingness` |
| 11 | Missingness policy (structural / MNAR) | `config/05_missingness.py` |
| 12 | Derivations (bins, flags, computed cols) | `config/06_derivations.py` |
| 13 | Schema validation (pandera) | `pandera` |
| 14 | Pre-imputation peek (free-form) | — |
| 15 | Imputation (MICE or simple) | `missingness_resolution` |
| 16 | Save + validate handoff datasets | `dataset_handoff` |

`config/07_analysis.py` and `config/08_report_settings.py` are used by `meningioma-modelling.ipynb`, not here.

</details>


## 00 · Setup

⚙️ Boots the pipeline and gives you a clean slate.

<details>
<summary>🔧 How it works</summary>

- 📦 Imports the pipeline modules (`schema_infer`, `cleaning`, `dda`, `missingness_resolution`, `dataset_handoff`).
- 🧹 Wipes and recreates `output/` so every run starts from scratch.
- 🔌 Exposes `load("NN_name")` to pull a numbered config module on demand.
- 📅 Sets `ANALYSIS_YEARS` as a default (`None` = all years); you re-set it in §03.

</details>

<details>
<summary>💡 Code examples &amp; scenarios</summary>

**Load a config module by number**
```python
_c01 = load("01_cohort")          # returns the module object
```

**Cohort year scope**
```python
ANALYSIS_YEARS = None             # 🌍 all years
ANALYSIS_YEARS = [2025]           # 📌 one cohort year
ANALYSIS_YEARS = [2024, 2025]     # 🗓️ a subset
```

</details>


In [1]:
import pandas as pd
pd.set_option("display.max_columns", None)
import pandera.pandas as pa

import numpy as np
import shutil
from pathlib import Path

from IPython.display import display

from schema_infer import (
    infer_schema, print_schema_template, print_column_uniques, schema_summary, ColSpec,
)
from cleaning import apply_schema, audit_duplicates, format_table_for_display
from dda import run_dda
from missingness_resolution import analyze_missingness, proper_mice_impute, rf_chained_impute, simple_impute_stage
from dataset_handoff import validate_handoff_datasets

OUTPUT_ROOT = Path("output")
if OUTPUT_ROOT.exists():
    shutil.rmtree(OUTPUT_ROOT)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

from config import load

#🟧🟧🟧 None = all years; e.g. [2025] for one cohort year

ANALYSIS_YEARS: list[int] | None = None


## 01 · Load raw data

📥 Reads the raw export into a dataframe — nothing else yet.

<details>
<summary>🔧 How it works · <code>config/01_cohort.py · load_raw</code></summary>

- 📄 `load_raw(DATA_PATH)` reads the export — **CSV or Excel**, picked automatically by file extension.
- 🔢 Prints the `rows × columns` count so you can confirm the file loaded.
- 🚫 No filtering or type coercion happens here.
- 👀 `df.head(0)` shows just the raw column headers, which feed the rename map in §02.

</details>

<details>
<summary>💡 Code examples &amp; scenarios</summary>

**CSV export**
```python
DATA_PATH = "Meningiomas PSKUS grants - Visi pacienti.csv"
df_raw = _c01.load_raw(DATA_PATH)
```

**Excel export** — same call, the extension decides the reader
```python
DATA_PATH = "cohort_export.xlsx"
df_raw = _c01.load_raw(DATA_PATH)
```

</details>


In [2]:
DATA_PATH = "Meningiomas PSKUS grants - Visi pacienti.csv"   # or "yourdata.csv"

_c01 = load("01_cohort")
df_raw = _c01.load_raw(DATA_PATH)
df_raw.head(0)


Loaded: 398 rows × 39 columns


,Nr.,Personas kods,Unnamed: 2,Vecums. gadi,"Dzimums. 0 - vīrietis\n1 - sieviete""",Histoloģija. 0 - nav\n1 - ir,WHO pakāpe (2021). 1 / 2 / 3,Progesterons. 0 - negatīvs\n1 - pozitīvs,Ki-67 (%). skaitlis. %,Smadzeņu parenhīmas invāzija. 0 - nav\n1 - ir,Nekroze histoloģiski. 0 - nav\n1 - ir,MRI izmeklējuma datums,Puse. 1 - labā\n2 - kreisā\n3 - viduslīnija,Lokalizācija: skull base / non–skull base. 0 - non-skull base\n1 - skull base,Cik meningiomas?,Max diametrs. skaitlis.cm,Tilpums,Pamatmodalitāte analīzei. 0 - MRI\n1 - CT\n3 - MRI+CT,K/v i/v. 0 - nav\n1 - ir,0 - primārs\n1 - recidīvs,Audzēja robeža. 1 = gluda. \n2 = neregulāra,Dural tail sign. 0 - nav\n1 - ir,Gredzenveida kontrastēšanās (tumor capsular enhancement). 0 - nav\n1 - ir,Kontrastēšanās veids. 0 - homogēna\n1 - heterogēna,Perifokāla tūska. 0 - nav\n1 - ir,Perifokālas tūskas tilpums. cm3,Masas efekts. 0 - nav\n1 - ir,Audzēja kalcifikācija. 0 - nav\n1 - ir,Cistiskas komponentes. 0 - nav\n1 - ir,Audzēja nekroze. 0 - nav\n1 - ir,Hemorāģiskas sastāvdaļas. 0 - nav\n1 - ir\n2 - nav skaidri izvērtējams,Kaule hiperostoze. 0 - nav\n1 - ir,Kaula invāzija (cortical destruction). 0 - nav\n1 - ir,Tumor Hyperintensity on DWI. 0 - nav\n1 - ir,Tumor Hyperintensity on T2. 0 - nav\n1 - ir,Tumor Hypointensity on T1. 0 - nav\n1 - ir,Sīnuss. 0 - neieaug\n1 - ieaug\n2 - ieaug un cauraug,Cauraug falx cerebri 0 - nav. 1 - ir,ADC map value


## 02 · Rename columns

🏷️ Turn messy raw headers into clean `snake_case` names.

1. ▶️ Run **see raw columns** → copy the printed skeleton
2. ✍️ Paste into `COLUMN_RENAME_MAP` and fill in the snake_case names
3. ✅ Run **apply rename**

<details>
<summary>🔧 How it works · <code>config/02_column_rename_map.py</code></summary>

- 🖨️ `list_cols(df_raw)` prints a copy-paste `COLUMN_RENAME_MAP` skeleton — one row per raw column.
- ✍️ You fill in the right-hand snake_case names.
- 🔁 `apply_rename(df_raw, COLUMN_RENAME_MAP)` returns the renamed frame.
- ⏱️ Renaming runs **before** schema inference, so every later step uses the clean names.

</details>

<details>
<summary>💡 Code examples &amp; scenarios</summary>

**Step 1 — print the skeleton**
```python
load("02_column_rename_map").list_cols(df_raw)
```

**Step 2 — fill it in**
```python
COLUMN_RENAME_MAP = {
    "Vecums. gadi": "age",
    "WHO pakāpe (2021). 1 / 2 / 3": "who_grade",
    # ... one entry per raw column
}
```

**Step 3 — apply**
```python
df_raw = load("02_column_rename_map").apply_rename(df_raw, COLUMN_RENAME_MAP)
df = df_raw
```

</details>


In [3]:
#🟧🟧🟧 Step 1 — see raw columns (run once per new dataset)

load("02_column_rename_map").list_cols(df_raw)


COLUMN_RENAME_MAP = {
    "Nr.": "",
    "Personas kods": "",
    "Unnamed: 2": "",
    "Vecums. gadi": "",
    "Dzimums. 0 - vīrietis\n1 - sieviete\"": "",
    "Histoloģija. 0 - nav\n1 - ir": "",
    "WHO pakāpe (2021). 1 / 2 / 3": "",
    "Progesterons. 0 - negatīvs\n1 - pozitīvs": "",
    "Ki-67 (%). skaitlis. %": "",
    "Smadzeņu parenhīmas invāzija. 0 - nav\n1 - ir": "",
    "Nekroze histoloģiski. 0 - nav\n1 - ir": "",
    "MRI izmeklējuma datums": "",
    "Puse. 1 - labā\n2 - kreisā\n3 - viduslīnija": "",
    "Lokalizācija: skull base / non–skull base. 0 - non-skull base\n1 - skull base": "",
    "Cik meningiomas?": "",
    "Max diametrs. skaitlis.cm": "",
    "Tilpums": "",
    "Pamatmodalitāte analīzei. 0 - MRI\n1 - CT\n3 - MRI+CT": "",
    "K/v i/v. 0 - nav\n1 - ir": "",
    "0 - primārs\n1 - recidīvs": "",
    "Audzēja robeža. 1 = gluda. \n2 = neregulāra": "",
    "Dural tail sign. 0 - nav\n1 - ir": "",
    "Gredzenveida kontrastēšanās (tumor capsular enhancement). 0 - nav\n

In [4]:
#🟧🟧🟧 Step 2 — paste skeleton here and fill in the right-hand names

COLUMN_RENAME_MAP = {
    "Nr.": "id",
    "Personas kods": "patient_code",
    "Unnamed: 2": "entry_year",
    "Vecums. gadi": "age",
    "Dzimums. 0 - vīrietis\n1 - sieviete\"": "sex",
    "Histoloģija. 0 - nav\n1 - ir": "histology_available",
    "WHO pakāpe (2021). 1 / 2 / 3": "who_grade",
    "Progesterons. 0 - negatīvs\n1 - pozitīvs": "progesterone_pos",
    "Ki-67 (%). skaitlis. %": "ki67_pct",
    "Smadzeņu parenhīmas invāzija. 0 - nav\n1 - ir": "brain_invasion",

    "Nekroze histoloģiski. 0 - nav\n1 - ir": "hist_necrosis",
    "MRI izmeklējuma datums": "mri_date",
    "Puse. 1 - labā\n2 - kreisā\n3 - viduslīnija": "side",
    "Lokalizācija: skull base / non–skull base. 0 - non-skull base\n1 - skull base": "tumor_location",
    "Cik meningiomas?": "meningioma_count",

    "Max diametrs. skaitlis.cm": "max_diameter_cm",
    "Tilpums": "tumor_volume",
    "Pamatmodalitāte analīzei. 0 - MRI\n1 - CT\n3 - MRI+CT": "additional_ct",
    "K/v i/v. 0 - nav\n1 - ir": "iv_contrast",
    "0 - primārs\n1 - recidīvs": "tumor_episode",
    "Audzēja robeža. 1 = gluda. \n2 = neregulāra": "tumor_margin",
    "Dural tail sign. 0 - nav\n1 - ir": "dural_tail",
    "Gredzenveida kontrastēšanās (tumor capsular enhancement). 0 - nav\n1 - ir": "capsular_enhancement",
    "Kontrastēšanās veids. 0 - homogēna\n1 - heterogēna": "heterogeneous_enhancement",
    "Perifokāla tūska. 0 - nav\n1 - ir": "perifocal_edema",

    "Perifokālas tūskas tilpums. cm3": "edema_volume_cm3",
    "Masas efekts. 0 - nav\n1 - ir": "mass_effect",
    "Audzēja kalcifikācija. 0 - nav\n1 - ir": "calcification",
    "Cistiskas komponentes. 0 - nav\n1 - ir": "cystic_component",
    "Audzēja nekroze. 0 - nav\n1 - ir": "mri_necrosis",
    "Hemorāģiskas sastāvdaļas. 0 - nav\n1 - ir\n2 - nav skaidri izvērtējams ": "hemorrhage",
    "Kaule hiperostoze. 0 - nav\n1 - ir": "hyperostosis",
    "Kaula invāzija (cortical destruction). 0 - nav\n1 - ir": "cortical_destruction",
    "Tumor Hyperintensity on DWI. 0 - nav\n1 - ir": "dwi_hyperintensity",
    "Tumor Hyperintensity on T2. 0 - nav\n1 - ir": "t2_hyperintensity",
    "Tumor Hypointensity on T1. 0 - nav\n1 - ir": "t1_hypointensity",

    "Sīnuss. 0 - neieaug\n1 - ieaug\n2 - ieaug un cauraug": "sinus_invasion",
    "Cauraug falx cerebri 0 - nav. 1 - ir": "transfalcine_extension",
    "ADC map value": "adc_value",
    }

In [5]:
#🟧🟧🟧 Step 3 — apply rename

df_raw = load("02_column_rename_map").apply_rename(df_raw, COLUMN_RENAME_MAP)
df = df_raw
df.head(0)

,id,patient_code,entry_year,age,sex,histology_available,who_grade,progesterone_pos,ki67_pct,brain_invasion,hist_necrosis,mri_date,side,tumor_location,meningioma_count,max_diameter_cm,tumor_volume,additional_ct,iv_contrast,tumor_episode,tumor_margin,dural_tail,capsular_enhancement,heterogeneous_enhancement,perifocal_edema,edema_volume_cm3,mass_effect,calcification,cystic_component,mri_necrosis,hemorrhage,hyperostosis,cortical_destruction,dwi_hyperintensity,t2_hyperintensity,t1_hypointensity,sinus_invasion,transfalcine_extension,adc_value


## 03 · Cohort filter

🗓️ Optionally trim the cohort to specific years. Use **renamed** names from §02.

<details>
<summary>🔧 How it works · <code>config/01_cohort.py · filter_cohort</code></summary>

- 🎯 `filter_cohort(df, YEAR_COLUMN, ANALYSIS_YEARS)` restricts the cohort to a set of years.
- 🌍 `None` → keep **all** years (just reports them).
- 📌 `[2025]` → keep one cohort year.
- 🗂️ `[2024, 2025]` → keep a subset.
- ⚠️ `[]` → **raises** (use `None` instead).
- 🔢 Prints rows kept vs dropped.

</details>

<details>
<summary>💡 Code examples &amp; scenarios</summary>

**Set the scope, then filter**
```python
YEAR_COLUMN = "entry_year"
ID_COLS = ["id", "patient_code", "entry_year"]

ANALYSIS_YEARS = None              # 🌍 all years
df_raw = _c01.filter_cohort(df_raw, YEAR_COLUMN, ANALYSIS_YEARS)
df = df_raw
```

**Other scenarios**
```python
ANALYSIS_YEARS = [2025]            # 📌 single year
ANALYSIS_YEARS = [2024, 2025]      # 🗂️ subset
ANALYSIS_YEARS = []                # ⚠️ ValueError — use None instead
```

</details>


In [6]:
YEAR_COLUMN = "entry_year"
ID_COLS = ["id", "patient_code", "entry_year"]

ANALYSIS_YEARS: list[int] | None = None   # e.g. [2025]; None = all years
# ANALYSIS_YEARS = 

In [7]:
df_raw = _c01.filter_cohort(df_raw, YEAR_COLUMN, ANALYSIS_YEARS)
df = df_raw

📅 Cohort · all years in entry_year
[2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026]
398 rows


## 04 · Schema

🧱 Decide what each column *is* (its `ColSpec` kind) before any coercion.

1. 🔎 Run **infer**
2. 🖨️ Run **print template**
3. 🧪 Run **column uniques** (nulls / replace hints)
4. ✍️ Edit **schema_overrides**
5. ✅ Run **apply overrides**

<details>
<summary>🔧 How it works · <code>config/03_schema_overrides.py</code></summary>

- 🤖 `infer_schema` guesses a `ColSpec` (kind + levels) per column.
- 🖨️ `print_schema_template` prints those guesses as an editable `schema_overrides` dict.
- ✍️ You correct the `kind`, `replace` maps, `nulls`, and `keep` flags.
- 🔗 `apply_schema_overrides(...)` merges your edits and writes `schema/schema_summary.csv`.
- ⏭️ Nothing is coerced yet — that happens in §06.

</details>

<details>
<summary>💡 Code examples &amp; scenarios</summary>

**Infer + print the editable template**
```python
schema = infer_schema(df_raw)
print_schema_template(schema)
```

**The `ColSpec` kinds you can declare**
```python
ColSpec(name="id",               kind="id")
ColSpec(name="age",              kind="continuous")
ColSpec(name="brain_invasion",   kind="binary")
ColSpec(name="sex",              kind="nominal",  replace={0: "male", 1: "female"})
ColSpec(name="who_grade",        kind="ordinal",  ordered_levels=["1", "2", "3"])
ColSpec(name="mri_date",         kind="datetime", datetime_bin="full", keep=False)
ColSpec(name="meningioma_count", kind="count")
ColSpec(name="progesterone_pos", kind="binary",   nulls=(2,))    # 🚫 map 2 → NaN
ColSpec(name="patient_code",     kind="id",        keep=False)   # 🗑️ drop after cleaning
```

**Merge + persist**
```python
load("03_schema_overrides").apply_schema_overrides(schema, schema_overrides, OUTPUT_ROOT)
```

</details>


In [8]:
schema = infer_schema(df_raw)
#schema_summary(schema)


In [9]:
print_schema_template(schema)


schema_overrides = {
    'id': ColSpec(name='id', kind='id'),
    'patient_code': ColSpec(name='patient_code', kind='id'),
    'entry_year': ColSpec(name='entry_year', kind='nominal'),
    'age': ColSpec(name='age', kind='continuous'),
    'sex': ColSpec(name='sex', kind='binary'),
    'histology_available': ColSpec(name='histology_available', kind='binary'),
    'who_grade': ColSpec(name='who_grade', kind='nominal'),
    'progesterone_pos': ColSpec(name='progesterone_pos', kind='binary'),
    'ki67_pct': ColSpec(name='ki67_pct', kind='text'),
    'brain_invasion': ColSpec(name='brain_invasion', kind='binary'),
    'hist_necrosis': ColSpec(name='hist_necrosis', kind='binary'),
    'mri_date': ColSpec(name='mri_date', kind='text'),
    'side': ColSpec(name='side', kind='nominal'),
    'tumor_location': ColSpec(name='tumor_location', kind='binary'),
    'meningioma_count': ColSpec(name='meningioma_count', kind='ordinal', ordered_levels=[1.0, 2.0, 3.0, 4.0, 5.0, 6.0]),
    'max_diameter_c

In [10]:
#🟧🟧🟧 Edit overrides, then run
schema_overrides = {
    'id': ColSpec(name='id', kind='id'),
    'patient_code': ColSpec(name='patient_code', kind="id"),
    
    'entry_year': ColSpec(name='entry_year', kind='count'),
    'age': ColSpec(name='age', kind='continuous'),
    'sex': ColSpec(name='sex', kind='nominal', replace={0:"male", 1:"female",}),
    
    'who_grade': ColSpec(name='who_grade', kind='ordinal', ordered_levels=["1","2","3"]),
    
    #'histology_available': ColSpec(name='histology_available', kind='binary', nulls=(2,)),
    'histology_available': ColSpec(name='histology_available', kind='skip'),

    'progesterone_pos': ColSpec(name='progesterone_pos', kind='binary', nulls=(2,)),
    'ki67_pct': ColSpec(name='ki67_pct', kind='text'),
    'brain_invasion': ColSpec(name='brain_invasion', kind='binary'),
    'hist_necrosis': ColSpec(name='hist_necrosis', kind='binary'),
    
    'mri_date': ColSpec(name='mri_date', kind='datetime', datetime_bin='full'),
    
    'side': ColSpec(name='side', kind='nominal', replace={'1': "right", '2': "left", '3': "midline"}),
    'tumor_location': ColSpec(name='tumor_location', kind='nominal', replace={0: "non_skull_base", 1: "skull_base"}, nulls=(2,)),
    'meningioma_count': ColSpec(name='meningioma_count', kind='count'),
    'max_diameter_cm': ColSpec(name='max_diameter_cm', kind='continuous'),
    'tumor_volume': ColSpec(name='tumor_volume', kind='continuous'),
    
    'additional_ct': ColSpec(name='additional_ct', kind='binary', replace={0.0: False, 1.0: pd.NA, 3.0: True}),
    'iv_contrast': ColSpec(name='iv_contrast', kind='binary'),

    'tumor_episode': ColSpec(name='tumor_episode', kind='ordinal', replace={'0': "primary", '1': "recurrent"}, ordered_levels=["primary", "recurrent"], nulls=("multiplas",)),
    
    'tumor_margin': ColSpec(name='tumor_margin', kind='nominal', replace={1: "regular", 2: "irregular"}, nulls=(0,)),
    'dural_tail': ColSpec(name='dural_tail', kind='binary'),
    
    'capsular_enhancement': ColSpec(name='capsular_enhancement', kind='binary'),
    'heterogeneous_enhancement': ColSpec(name='heterogeneous_enhancement', kind='binary'),
    'dwi_hyperintensity': ColSpec(name='dwi_hyperintensity', kind='binary', nulls=('-',)),
    't2_hyperintensity': ColSpec(name='t2_hyperintensity', kind='binary', nulls=('-',)),
    't1_hypointensity': ColSpec(name='t1_hypointensity', kind='binary', nulls=('-',)),
    
    'perifocal_edema': ColSpec(name='perifocal_edema', kind='binary'),
    'edema_volume_cm3': ColSpec(name='edema_volume_cm3', kind='continuous'),
    
    'mass_effect': ColSpec(name='mass_effect', kind='binary'),
    'calcification': ColSpec(name='calcification', kind='binary'),
    'cystic_component': ColSpec(name='cystic_component', kind='binary'),
    'mri_necrosis': ColSpec(name='mri_necrosis', kind='binary'),
    'hemorrhage': ColSpec(name='hemorrhage', kind='binary', nulls=(2.0,)),
    'hyperostosis': ColSpec(name='hyperostosis', kind='binary'),
    'sinus_invasion': ColSpec(name='sinus_invasion', kind='ordinal', replace={0: "no_invasion", 1: "sinus_invasion", 2: "transsinus_extension"}, ordered_levels=["no_invasion", "sinus_invasion", "transsinus_extension"]),
    'cortical_destruction': ColSpec(name='cortical_destruction', kind='binary'),
    'transfalcine_extension': ColSpec(name='transfalcine_extension', kind='binary'),
    
    'adc_value': ColSpec(name='adc_value', kind='continuous'),
    }

In [11]:
#df.columns.tolist()

In [12]:
load("03_schema_overrides").apply_schema_overrides(schema, schema_overrides, OUTPUT_ROOT)

## 05 · Row filters — pre-schema

🧹 Drop excluded rows **on raw strings**, before typing — so they never become categorical levels (e.g. spinal meningioma in `side` / `mri_date`).

<details>
<summary>🔧 How it works · <code>config/04_row_filters.py</code></summary>

- 🏷️ A `RowFilter` is a named `keep(df) → bool mask` rule with a `note`.
- 🔀 `active=False` skips a filter but still logs it.
- ▶️ `apply_row_filters` runs the list in order, returning the filtered frame + a rows-before/after/removed log.
- ⏱️ Run these **pre-schema** so dropped rows never define categorical levels later.
- ➡️ The post-schema pass (after typing) is §08.

</details>

<details>
<summary>💡 Code examples &amp; scenarios</summary>

**Built-in inclusion filter (drop spinal cases)**
```python
_c04 = load("04_row_filters")

pre_schema_row_filters = [
    _c04.brain_meningioma_row_filter(),               # ✅ active by default
    # _c04.brain_meningioma_row_filter(active=False),  # 🔀 skip but still log
]

df, pre_schema_row_filter_log = _c04.apply_row_filters(df, pre_schema_row_filters)
n_rows_pre_schema = len(df)
```

**Custom raw-string filter**
```python
_c04.RowFilter(
    name="non-empty side",
    keep=lambda d: d["side"].astype("string").str.strip().ne(""),
    note="drop rows with blank laterality",
    active=True,
)
```

</details>


In [13]:
_c04 = load("04_row_filters")

pre_schema_row_filters = [
    _c04.brain_meningioma_row_filter(),
]

df, pre_schema_row_filter_log = _c04.apply_row_filters(
    df, pre_schema_row_filters,
)
n_rows_pre_schema = len(df)
pre_schema_row_filter_log

,name,active,rows_before,rows_after,rows_removed,note
0,Meningioma location - brain,True,398,394,4,inclusion criteria - brain meningioma


## 06 · Apply schema

🔧 Coerce every column to the type you declared in §04.

<details>
<summary>🔧 How it works · <code>cleaning.apply_schema</code></summary>

- 🔁 Applies each column's `replace` map.
- 🧮 Casts to the right dtype — datetime, ordered categorical, numeric, etc.
- 🚫 Maps declared `nulls` to `NaN`.
- 🗑️ Drops columns flagged `keep=False`.
- 📝 `schema_log` records every action for the report.

</details>

<details>
<summary>💡 Code examples &amp; scenarios</summary>

**Coerce + capture the audit log**
```python
schema_log = []
df = apply_schema(df, schema, log=schema_log)
n_rows_after_schema = len(df)
df.head()
```

</details>


In [14]:
schema_log = []
df = apply_schema(df, schema, log=schema_log)
n_rows_after_schema = len(df)
df.head()

,id,patient_code,entry_year,age,sex,who_grade,progesterone_pos,ki67_pct,brain_invasion,hist_necrosis,mri_date,side,tumor_location,meningioma_count,max_diameter_cm,tumor_volume,additional_ct,iv_contrast,tumor_episode,tumor_margin,dural_tail,capsular_enhancement,heterogeneous_enhancement,perifocal_edema,edema_volume_cm3,mass_effect,calcification,cystic_component,mri_necrosis,hemorrhage,hyperostosis,cortical_destruction,dwi_hyperintensity,t2_hyperintensity,t1_hypointensity,sinus_invasion,transfalcine_extension,adc_value
0,1.0,290357-12753,2025,67.0,female,NaN,<NA>,<NA>,<NA>,<NA>,NaT,NaN,NaN,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,NaN,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,<NA>,<NA>
1,2.0,070458-11352,2025,67.0,female,1,True,1-3,False,False,2025-06-27,right,skull_base,2,4.9,36.5,False,True,primary,regular,False,True,False,True,5.0,True,True,False,False,False,False,False,True,True,True,no_invasion,False,0.88
2,3.0,230949-11093,2025,76.0,female,1,True,1-5,False,False,2025-09-05,midline,skull_base,1,2.8,6.86,True,True,primary,irregular,False,True,False,True,26.0,True,False,False,False,False,False,False,True,True,True,no_invasion,True,0.94
3,4.0,140352-11498,2025,73.0,female,2,True,25-30,True,True,2025-07-23,right,non_skull_base,1,4.7,40.9,True,True,recurrent,irregular,False,True,True,True,135.0,True,True,True,False,True,False,False,True,True,True,no_invasion,False,0.6
4,5.0,151269-12200,2025,55.0,male,1,True,1-2,False,False,2025-08-04,right,non_skull_base,1,3.7,8.3,True,True,primary,irregular,True,True,False,True,24.0,True,True,False,False,False,False,False,True,True,True,no_invasion,False,1.2


## 07 · Duplicate audit

🕵️ Find rows that share the same identifier(s).

<details>
<summary>🔧 How it works · <code>cleaning.audit_duplicates</code></summary>

- 👥 Groups rows sharing the same `id_cols` and returns the duplicate groups for review.
- 👀 `drop=False` → report only (default, safe).
- ✂️ `drop=True` → actually remove duplicates (use only once confirmed real).
- 1️⃣ `include_first=True` keeps the first occurrence visible in the report.

</details>

<details>
<summary>💡 Code examples &amp; scenarios</summary>

**Report only (recommended first pass)**
```python
dupes, df = audit_duplicates(df, id_cols=ID_COLS, include_first=True, drop=False)
dupes.head() if len(dupes) else print("No duplicate groups found.")
```

**Drop confirmed duplicates**
```python
dupes, df = audit_duplicates(df, id_cols=ID_COLS, drop=True)
```

</details>


In [15]:
dupes, df = audit_duplicates(df, id_cols=ID_COLS, include_first=True, drop=False)
dupes.head() if len(dupes) else print('No duplicate groups found.')

No duplicate groups found.


## 08 · Row filters — post-schema

🧹 Same machinery as §05, but **after** typing — rules can use real dtypes. Toggle `active`, run, then finalize.

<details>
<summary>🔧 How it works · <code>config/04_row_filters.py</code></summary>

- 🧱 Runs **after** `apply_schema`, so filters can use coerced types (datetimes, ordered categories).
- ✅ Typical inclusion rules: "`who_grade` exists", "MRI exists".
- 🔗 `combine_row_filter_logs` merges the pre- and post-schema logs in run order.
- 💾 `finalize_row_drops` writes the cleaning artifacts (`cleaning_summary.csv`, drop log) for the report.

</details>

<details>
<summary>💡 Code examples &amp; scenarios</summary>

**Declare + apply post-schema filters**
```python
post_schema_row_filters = [
    _c04.RowFilter(name="who_grade exists", keep=lambda d: d["who_grade"].notna(),
                   note="inclusion criteria - histological WHO grade", active=True),
    _c04.RowFilter(name="MRI exists", keep=lambda d: d["mri_date"].notna(),
                   note="inclusion criteria - MRI", active=True),
]

df, post_schema_row_filter_log = _c04.apply_row_filters(df, post_schema_row_filters)
```

**More scenarios (toggle with `active`)**
```python
_c04.RowFilter(name="adult patients only", keep=lambda d: d["age"] >= 18,
               note="age ≥ 18", active=False)                        # 🔀 off
_c04.RowFilter(name="exclude WHO grade 2 or 3",
               keep=lambda d: ~d["who_grade"].isin(["2", "3"]),
               note="grade 1 only", active=False)
```

**Merge logs + finalize**
```python
row_filter_log = _c04.combine_row_filter_logs(
    pre_schema_row_filter_log, post_schema_row_filter_log,
)
df = _c04.finalize_row_drops(
    df, row_filter_log, output_root=OUTPUT_ROOT, df_raw=df_raw,
    n_rows_pre_schema=n_rows_pre_schema, n_rows_after_schema=n_rows_after_schema,
    schema=schema, dupes=dupes, schema_log=schema_log,
)
```

</details>


In [16]:
post_schema_row_filters = [
    _c04.RowFilter(
        name="who_grade exists",
        keep=lambda d: d["who_grade"].notna(),
        note="inclusion criteria - histological WHO grade",
        active=True,
    ),
    _c04.RowFilter(
        name="MRI exists",
        keep=lambda d: d["mri_date"].notna(),
        note="inclusion criteria - MRI",
        active=True,
    ),
    # _c04.RowFilter(
    #     name="sex known",
    #     keep=lambda d: d["sex"] != "unknown",
    #     note="Keep rows where sex is known (not 'unknown')",
    #     active=False,
    # ),
    # _c04.RowFilter(
    #     name="adult patients only",
    #     keep=lambda d: d["age"] >= 18,
    #     note="Keep rows where age is 18 or older",
    #     active=False,
    # ),
    # _c04.RowFilter(
    #     name="exclude WHO grade 2 or 3",
    #     keep=lambda d: ~d["who_grade"].isin(["2", "3"]),
    #     note="Keep rows where WHO grade is 1 (exclude grades 2 and 3)",
    #     active=False,
    # ),
]

df, post_schema_row_filter_log = _c04.apply_row_filters(df, post_schema_row_filters)
row_filter_log = _c04.combine_row_filter_logs(
    pre_schema_row_filter_log,
    post_schema_row_filter_log,
)
row_filter_log


,name,active,rows_before,rows_after,rows_removed,note
0,Meningioma location - brain,True,398,394,4,inclusion criteria - brain meningioma
1,who_grade exists,True,394,361,33,inclusion criteria - histological WHO grade
2,MRI exists,True,361,352,9,inclusion criteria - MRI


In [17]:
df = _c04.finalize_row_drops(
    df, row_filter_log,
    output_root=OUTPUT_ROOT,
    df_raw=df_raw,
    n_rows_pre_schema=n_rows_pre_schema,
    n_rows_after_schema=n_rows_after_schema,
    schema=schema,
    dupes=dupes,
    schema_log=schema_log,
)


,name,active,rows_before,rows_after,rows_removed,note
0,Meningioma location - brain,True,398,394,4,inclusion criteria - brain meningioma
1,who_grade exists,True,394,361,33,inclusion criteria - histological WHO grade
2,MRI exists,True,361,352,9,inclusion criteria - MRI


## 09 · DDA — first pass

📊 A descriptive sanity pass on the typed cohort.

<details>
<summary>🔧 How it works · <code>dda.run_dda</code></summary>

- 📐 Profiles overall shape, missing-cell %, and per-column summaries.
- 🖼️ Writes DDA tables and figures under `output/dda/`.
- 🛡️ Pure read-only checkpoint before any missingness decisions.
- ♻️ Does **not** change `df`.

</details>

<details>
<summary>💡 Code examples &amp; scenarios</summary>

**Run the descriptive pass**
```python
dda_tables = run_dda(df, schema, output_root=OUTPUT_ROOT)
```

</details>


In [18]:
dda_tables = run_dda(df, schema, output_root=OUTPUT_ROOT)

for name in ("overall", "continuous", "categorical", "binary", "datetime", "id_text"):
    print(f"\n--- {name} ---")
    tbl = dda_tables.get(name)
    if tbl is None or tbl.empty:
        print("(none)")
    else:
        display(format_table_for_display(tbl))



--- overall ---


,n_rows,n_cols,n_cols_analysed,missing_cells_pct
0,352,38,38,0.7



--- continuous ---


,column,kind,n,n_unique,missing_pct,min,p_5th,median,mean,trimmed_mean,p_95th,max,mode,std,cv,iqr,skewness,kurtosis
0,entry_year,count,352,9,0.0,2018.00,2018.00,2023.00,2021.87,2021.96,2025.00,2026.0,2025.00,2.64,0.00,5.00,-0.22,-1.54
1,age,continuous,352,60,0.0,20.00,40.00,65.00,63.11,63.70,81.00,92.0,71.00,12.68,0.20,17.25,-0.43,-0.14
2,meningioma_count,count,352,6,0.0,1.00,1.00,1.00,1.17,1.02,2.00,6.0,1.00,0.58,0.50,0.00,4.51,24.42
3,max_diameter_cm,continuous,352,88,0.0,0.20,1.79,3.80,4.06,3.95,7.30,9.2,1.80,1.73,0.43,2.50,0.51,-0.42
4,tumor_volume,continuous,329,282,6.5,0.30,1.60,14.70,27.62,21.57,98.76,168.0,2.00,31.80,1.15,31.42,1.66,2.31
5,edema_volume_cm3,continuous,333,182,5.4,0.00,0.00,4.48,20.85,13.43,93.60,197.0,0.00,32.74,1.57,29.20,2.16,5.06
6,adc_value,continuous,309,77,12.2,0.41,0.64,0.82,0.85,0.83,1.19,1.7,0.79,0.17,0.20,0.17,1.19,2.99



--- categorical ---


...

,column,kind,ordered,n,n_unique,missing_pct,mode,mode_pct,rarest,rarest_pct,max_class_imbalance,balance,entropy_bin
0,progesterone_pos,binary,False,351,2,0.3,True,97.4,False,2.6,38.00,0.17,0.17
1,brain_invasion,binary,False,352,2,0.0,False,98.3,True,1.7,57.67,0.12,0.12
2,hist_necrosis,binary,False,352,2,0.0,False,90.1,True,9.9,9.06,0.47,0.47
3,additional_ct,binary,False,352,2,0.0,False,77.6,True,22.4,3.46,0.77,0.77
4,iv_contrast,binary,False,352,2,0.0,True,96.9,False,3.1,31.00,0.20,0.20
5,dural_tail,binary,False,352,2,0.0,True,81.2,False,18.8,4.33,0.70,0.70
6,capsular_enhancement,binary,False,352,2,0.0,True,88.1,False,11.9,7.38,0.53,0.53
7,heterogeneous_enhancement,binary,False,352,2,0.0,True,58.8,False,41.2,1.43,0.98,0.98
8,perifocal_edema,binary,False,352,2,0.0,True,65.3,False,34.7,1.89,0.93,0.93
9,mass_effect,binary,False,352,2,0.0,True,86.1,False,13.9,6.18,0.58,0.58



--- datetime ---


,column,kind,n,missing_pct,min,max,span_days
0,mri_date,datetime,352,0,2017-10-23,2025-12-27,2987



--- id_text ---


,column,kind,n,missing_pct,n_unique
0,id,id,352,0,352
1,patient_code,id,352,0,352
2,ki67_pct,text,352,0,42


## 10 · Missingness story

🕳️ Quantify what's missing, before deciding what to do about it.

<details>
<summary>🔧 How it works · <code>analyze_missingness</code></summary>

- 📉 `analyze_missingness(df, output_root=OUTPUT_ROOT)` reports missingness per column (count + %).
- 🖼️ Saves the missingness tables/figures under `output/missingness/`.
- 🔍 Purely diagnostic — tells you *which* columns are missing and *how much*.
- ➡️ You decide the actual policy in §11.

</details>

<details>
<summary>💡 Code examples &amp; scenarios</summary>

**Summarise + show only the columns with gaps**
```python
missing_summary = analyze_missingness(df, output_root=OUTPUT_ROOT)
missing_summary[missing_summary.n_missing > 0]
```

</details>


In [19]:
missing_summary = analyze_missingness(df, output_root=OUTPUT_ROOT)
missing_summary[missing_summary.n_missing > 0]

,column,n_missing,pct_missing
0,adc_value,43,12.22
1,tumor_volume,23,6.53
2,edema_volume_cm3,19,5.40
3,dwi_hyperintensity,4,1.14
4,hemorrhage,3,0.85
5,t2_hyperintensity,2,0.57
6,t1_hypointensity,2,0.57
7,cortical_destruction,1,0.28
8,hyperostosis,1,0.28
9,progesterone_pos,1,0.28


## 11 · Missingness policy

🧭 Tell the pipeline *how* to treat missing values — declare it, then apply once.

<details>
<summary>🔧 How it works · <code>config/05_missingness.py</code></summary>

Two declarative knobs decide what gets imputed in §15:

- 🧩 **`StructuralGroup`** — slot-style columns where NaN means *the slot does not exist*. Instead of imputing, it derives a count/max and marks the raw columns `skip`.
- 🚩 **`MnarColumn`** — missing-not-at-random: missingness itself may be informative, so it adds a binary `<col>_missing` flag to the schema.
- ▶️ `apply_missingness_policy` applies both and returns updated `df`, `schema`, and an audit log.
- 🪹 Both lists are empty by default — add entries only when a column genuinely fits.

</details>

<details>
<summary>💡 Code examples &amp; scenarios</summary>

**Structural group — derive count/max, skip the raw slots**
```python
_c05 = load("05_missingness")

STRUCTURAL_GROUPS = [
    _c05.StructuralGroup(
        name="lesion_mri_pirads",
        cols=["lesion_1_MRI_PIRADS", "lesion_2_MRI_PIRADS", "lesion_3_MRI_PIRADS"],
        derive_count_col="n_mri_pirads_lesions",
        derive_max_col="max_mri_pirads",
        skip_raw=True,
        reason="Blank lesion slots mean the lesion does not exist, not unknown.",
    ),
]
```

**MNAR column — add an informative `<col>_missing` flag**
```python
MNAR_COLUMNS = [
    _c05.MnarColumn(
        col="ki67_pct", flag_col="ki67_pct_missing",
        reason="Ki-67 may be absent because it was not measured/reported.",
    ),
]
```

**Apply both (empty lists are fine — nothing happens)**
```python
df, schema, missingness_log = _c05.apply_missingness_policy(
    df=df, schema=schema,
    structural_groups=STRUCTURAL_GROUPS,
    mnar_columns=MNAR_COLUMNS,
)
```

</details>


In [20]:
_c05 = load("05_missingness")

STRUCTURAL_GROUPS = [
    # Example only. Keep empty if we do not currently have slot-style columns.
    # _c05.StructuralGroup(
    #     name="lesion_mri_pirads",
    #     cols=["lesion_1_MRI_PIRADS", "lesion_2_MRI_PIRADS", "lesion_3_MRI_PIRADS"],
    #     derive_count_col="n_mri_pirads_lesions",
    #     derive_max_col="max_mri_pirads",
    #     skip_raw=True,
    #     reason="Blank lesion slots mean lesion does not exist, not unknown.",
    # ),
]

MNAR_COLUMNS = [
    # Add only when missingness itself may be informative.
    # _c05.MnarColumn(
    #     col="ki67_pct",
    #     flag_col="ki67_pct_missing",
    #     reason="Ki-67 may be absent because it was not measured/reported in selected cases.",
    # ),
    # _c05.MnarColumn(
    #     col="adc_value",
    #     flag_col="adc_value_missing",
    #     reason="ADC may be absent when DWI/ADC was unavailable or non-diagnostic.",
    # ),
]


In [21]:
df, schema, missingness_log = _c05.apply_missingness_policy(
    df=df,
    schema=schema,
    structural_groups=STRUCTURAL_GROUPS,
    mnar_columns=MNAR_COLUMNS,
)

missingness_log

,policy,name,requested_cols,available_cols,missing_cols,created_cols,schema_action,reason,status


## 12 · Derivations (pre-imputation)

🧬 Build new analysis columns from the cleaned ones — declare them in `DERIVATIONS` (no `.py` edits needed).

<details>
<summary>🔧 How it works · <code>config/06_derivations.py</code></summary>

All study-specific logic lives in the notebook; `06_derivations.py` is just the engine. Three building blocks:

- 📊 **`BinNumeric`** — cut a numeric column into ordered bins (`age` → `age_bins`). Bins = edge values, labels = one per gap (`len(bins) - 1 == len(labels)`). Default `right=False` gives left-closed intervals (`[50, 60)` → `"50-59"`).
- 🔧 **`Apply`** — single-column custom logic via `fn=lambda s: ...` (Ki-67 midpoint, grouped labels, boolean flags).
- 🧮 **`Compute`** — needs several columns; `fn` receives the whole frame (e.g. zeroing `edema_volume_cm3` when there is no perifocal edema).

Per-entry switches:

- 🔀 `active=False` skips an entry.
- ♻️ `overwrite=True` replaces an existing column.
- ▶️ `apply_derivations` runs the list, updates the schema, previews new columns, and (with `write_csv=True`) writes the cleaned dataset + derivation log.
- 🔁 `DERIVED_DEPENDENCIES` + `apply_post_mice_derivations` re-apply the same rules after R MICE in §15.

</details>

<details>
<summary>💡 Code examples &amp; scenarios</summary>

**`BinNumeric` — ordered age bands**
```python
_c06 = load("06_derivations")

_c06.BinNumeric(
    name="age_bins", source="age",
    bins=[-np.inf, 50, 60, 70, 80, np.inf],
    labels=["<50", "50-59", "60-69", "70-79", "80+"],
    kind="ordinal", reason="Age groups for descriptive tables.",
)
```

**`Apply` — boolean flag and an ordinal group**
```python
_c06.Apply(name="high_grade", source="who_grade",
           fn=lambda s: s.astype("Float64").pipe(lambda sf: (sf == 2) | (sf == 3)),
           kind="binary", reason="WHO grade 2/3 = high-grade.")

_c06.Apply(name="ki67_group", source="ki67_mid", fn=lambda s: s.map(_ki67_group),
           kind="ordinal", ordered_levels=["low_le_4", "intermediate_5_9", "high_ge_10"])
```

**`Compute` — multi-column rule (overwrite in place)**
```python
_c06.Compute(
    name="edema_volume_cm3", sources=["perifocal_edema", "edema_volume_cm3"],
    fn=lambda d: d["edema_volume_cm3"].mask(
        d["perifocal_edema"].fillna(True).astype(float) == 0, 0),
    kind="continuous", overwrite=True,
    reason="No perifocal edema => edema volume is structurally 0, not missing.",
)
```

**Run the list**
```python
df, schema, derivation_log = _c06.apply_derivations(
    df=df, schema=schema, derivations=DERIVATIONS,
    output_root=OUTPUT_ROOT, write_csv=True,
)
```

</details>


In [22]:
_c06 = load("06_derivations")


def _ki67_midpoint(x):
    if pd.isna(x):
        return pd.NA
    parts = str(x).replace(",", ".").split("-")
    nums = [float(p) for p in parts]
    return sum(nums) / len(nums)
def _ki67_group(x):
    if pd.isna(x):
        return pd.NA
    if x <= 4:
        return "low_le_4"
    if x < 10:
        return "intermediate_5_9"
    return "high_ge_10"

DERIVATIONS = [
    _c06.BinNumeric(
        name="age_bins",
        source="age",
        bins=[-np.inf, 50, 60, 70, 80, np.inf],
        labels=["<50", "50-59", "60-69", "70-79", "80+"],
        kind="ordinal",
        active=True,
        overwrite=False,
        reason="Age groups for descriptive tables.",
    ),
    _c06.Apply(
        name="high_grade",
        source="who_grade",
        fn=lambda s: s.astype("Float64").pipe(lambda sf: (sf == 2) | (sf == 3)),
        kind="binary",
        active=True,
        overwrite=False,
        reason="WHO grade 2/3 = high-grade meningioma.",
    ),
    _c06.Apply(
        name="multiple_meningiomas",
        source="meningioma_count",
        fn=lambda s: s.astype("Float64") > 1,
        kind="binary",
        active=True,
        overwrite=False,
        reason=">1 meningioma = multiple",
    ),
    _c06.Apply(
        name="ki67_mid",
        source="ki67_pct",
        fn=lambda s: s.map(_ki67_midpoint).astype("Float64"),
        kind="continuous",
        active=True,
        overwrite=False,
        reason="Midpoint of Ki-67 range strings.",
    ),
    _c06.Apply(
        name="ki67_group",
        source="ki67_mid",
        fn=lambda s: s.map(_ki67_group),
        kind="ordinal",
        ordered_levels=["low_le_4", "intermediate_5_9", "high_ge_10"],
        active=True,
        overwrite=False,
        reason="Ki-67 clinical groups: ≤4 / 5-9 / ≥10.",
    ),
    _c06.Compute(
        name="edema_volume_cm3",
        sources=["perifocal_edema", "edema_volume_cm3"],
        fn=lambda d: d["edema_volume_cm3"].mask(
            d["perifocal_edema"].fillna(True).astype(float) == 0, 0
        ),
        kind="continuous",
        active=True,
        overwrite=True,
        reason="No perifocal edema => edema volume is structurally 0, not missing.",
    ),
    ]

# Derived parent->child map for MICE (see §12 notes): each derived column lists
# its source(s) so they can be recreated after imputation. edema_volume_cm3 is an
# in-place adjustment of an imputed variable, not a pure derived column, so it is
# intentionally excluded here.
DERIVED_DEPENDENCIES = {
    "age_bins": ["age"],
    "high_grade": ["who_grade"],            # analysis outcome (kept as predictor)
    "multiple_meningiomas": ["meningioma_count"],
    "ki67_mid": ["ki67_pct"],
    "ki67_group": ["ki67_mid"],
}


def apply_post_mice_derivations(frame):
    # df -> df wrapper around the single source-of-truth derivation engine.
    # Recreates derived columns from imputed sources after R MICE, reusing the
    # same DERIVATIONS list as cleaning so clinical rules are never duplicated.
    out, _schema, _log = _c06.apply_derivations(
        frame, schema, DERIVATIONS, preview=False,
    )
    return out


In [23]:
df, schema, derivation_log = _c06.apply_derivations(
    df=df,
    schema=schema,
    derivations=DERIVATIONS,
    output_root=OUTPUT_ROOT,
    write_csv=True,
)

derivation_log

,age_bins,high_grade,multiple_meningiomas,ki67_mid,ki67_group
1,60-69,False,True,2.0,low_le_4
2,70-79,False,False,3.0,low_le_4
3,70-79,True,False,27.5,high_ge_10
4,50-59,False,False,1.5,low_le_4
5,50-59,False,False,1.5,low_le_4


,derivation,type,active,source,kind,rows_nonmissing,rows_missing,schema_action,warning,reason
0,age_bins,BinNumeric,True,age,ordinal,352,0,added ColSpec (ordinal) for age_bins,,Age groups for descriptive tables.
1,high_grade,Apply,True,who_grade,binary,352,0,added ColSpec (binary) for high_grade,,WHO grade 2/3 = high-grade meningioma.
2,multiple_meningiomas,Apply,True,meningioma_count,binary,352,0,added ColSpec (binary) for multiple_meningiomas,,>1 meningioma = multiple
3,ki67_mid,Apply,True,ki67_pct,continuous,352,0,added ColSpec (continuous) for ki67_mid,,Midpoint of Ki-67 range strings.
4,ki67_group,Apply,True,ki67_mid,ordinal,352,0,added ColSpec (ordinal) for ki67_group,,Ki-67 clinical groups: ≤4 / 5-9 / ≥10.
5,edema_volume_cm3,Compute,True,"perifocal_edema, edema_volume_cm3",continuous,333,19,updated ColSpec (continuous) for edema_volume_cm3,,No perifocal edema => edema volume is structur...


## 13 · Schema validation

✅ Fail loudly if any column drifted from what §04 declared.

<details>
<summary>🔧 How it works · <code>pandera</code></summary>

- 🧱 Builds an explicit `pandera` `DataFrameSchema` from the declared kinds.
- 🔎 Checks category membership + ordering, numeric ranges, and nullability.
- 🚨 `strict=True` → any unexpected/dropped/extra column fails the check.
- 🔁 The same `schema_validation` is reused in §15 to check **every** imputed draw, not just the first.

</details>

<details>
<summary>💡 Code examples &amp; scenarios</summary>

**Helper for categorical checks (ordered or not)**
```python
def category_validation(expected, ordered=False):
    expected = tuple(expected)
    if ordered:
        return (pa.Check.isin(expected),
                pa.Check(lambda s: s.cat.ordered),
                pa.Check(lambda s: tuple(s.cat.categories) == expected))
    return (pa.Check.isin(expected),)
```

**Declare + run the schema**
```python
schema_validation = pa.DataFrameSchema(
    {
        "sex":       pa.Column("category", checks=[*category_validation(("male", "female"))]),
        "who_grade": pa.Column("category", checks=[*category_validation(("1", "2", "3"), ordered=True)]),
        # ... one Column per kept column
    },
    strict=True,
)
schema_validation(df, lazy=True).head()
```

**Validate every imputed draw (§15)**
```python
for _frame in imputed_frames:
    schema_validation(_frame, lazy=True)
```

</details>

In [24]:
#df.columns

In [25]:
#🟧🟧🟧 HELPERS
def category_validation(expected, ordered=False):
    expected = tuple(expected)
    if ordered:
        return (
            pa.Check.isin(expected),
            pa.Check(lambda s: s.cat.ordered),
            pa.Check(lambda s: tuple(s.cat.categories) == expected)
            )
    
    return (
        pa.Check.isin(expected),
        pa.Check(
            lambda s: set(expected).issubset(
                set(
                    s.dropna().unique()
                )
            )
        )
    )
#===================================================================

schema_validation = pa.DataFrameSchema(
    {
        'id': pa.Column(dtype=str, nullable=False, unique=True),
        'patient_code': pa.Column(dtype=str, unique=True),
        'entry_year': pa.Column(dtype="Int64"),
        'age': pa.Column(
            dtype="Float64", 
            nullable=True,
            checks=[
                pa.Check.ge(18),
                pa.Check.le(120)
            ]
            ),
        'sex': pa.Column(
            dtype="category",
            nullable=True,
            checks=[*category_validation(("male", "female"))]
            ),
        
        #'histology_available': pa.Column(dtype="boolean",nullable=True,),

        'who_grade': pa.Column(
                dtype="category",
                nullable=False,
                checks=[*category_validation(("1", "2", "3"), ordered=True)]
                ),
        'progesterone_pos': pa.Column(
                dtype="boolean",
                nullable=True,
                ),
        'ki67_pct': pa.Column(
                dtype=str,
                nullable=True
                ),
        'brain_invasion': pa.Column(
                dtype="boolean",
                nullable=True,
                ),
        'hist_necrosis': pa.Column(
                dtype="boolean",
                nullable=True,
                ),
        
        'mri_date': pa.Column(
                dtype="datetime64",
                nullable=False,
                checks=[
                    pa.Check.ge(pd.Timestamp("2017-01-01")),
                    pa.Check.le(pd.Timestamp.today())
                ]),
        
        'side': pa.Column(
                dtype="category",
                nullable=True,
                checks=[*category_validation(("right", "midline", "left"))]
                ),
        'tumor_location': pa.Column(
                dtype="category",
                nullable=True,
                checks=[*category_validation(("non_skull_base", "skull_base"))]
                ),
        'meningioma_count': pa.Column(
                dtype="Int64",
                nullable=False
                ),
        'max_diameter_cm': pa.Column(
                dtype="Float64",
                nullable=True
                ),
        'tumor_volume': pa.Column(
                dtype="Float64",
                nullable=True
                ),
        
        'additional_ct': pa.Column(
                dtype="boolean",
                nullable=True,
                ),
        'iv_contrast': pa.Column(
                dtype="boolean",
                nullable=True,
                ),
        
        'tumor_episode': pa.Column(
                dtype="category",
                nullable=True,
                checks=[*category_validation(("primary", "recurrent"), ordered=True)]
                ),
        'tumor_margin': pa.Column(
                dtype="category",
                nullable=True,
                checks=[*category_validation({"regular", "irregular"})]
                ),
        'dural_tail': pa.Column(
                dtype="boolean",
                nullable=True,
                ),
        'capsular_enhancement': pa.Column(
                dtype="boolean",
                nullable=True,
                ),
        'heterogeneous_enhancement': pa.Column(
                dtype="boolean",
                nullable=True,
                ),
        
        'perifocal_edema': pa.Column(
                dtype="boolean",
                nullable=True,
                ),
        'edema_volume_cm3': pa.Column(
                dtype="Float64",
                nullable=True
                ),
        
        'mass_effect': pa.Column(
                dtype="boolean",
                nullable=True,
                ),
        'calcification': pa.Column(
                dtype="boolean",
                nullable=True,
                ),
        'cystic_component': pa.Column(
                dtype="boolean",
                nullable=True,
                ),
        'mri_necrosis': pa.Column(
                dtype="boolean",
                nullable=True,
                ),
        'hemorrhage': pa.Column(
                dtype="boolean",
                nullable=True,
                ),
        'hyperostosis': pa.Column(
                dtype="boolean",
                nullable=True,
                ),
        'cortical_destruction': pa.Column(
                dtype="boolean",
                nullable=True,
                ),
        
        'dwi_hyperintensity': pa.Column(
                dtype="boolean",
                nullable=True,
                ),
        't2_hyperintensity': pa.Column(
                dtype="boolean",
                nullable=True,
                ),
        't1_hypointensity': pa.Column(
                dtype="boolean",
                nullable=True,
                ),
        
        'sinus_invasion': pa.Column(
                dtype="category",
                nullable=True,
                checks=[*category_validation(("no_invasion", "sinus_invasion", "transsinus_extension"), ordered=True)]
                ),
        'transfalcine_extension': pa.Column(
                dtype="boolean",
                nullable=True,
                ),
        'adc_value': pa.Column(
            dtype="Float64",
            nullable=True
            ),
    

        'age_bins': pa.Column(
            dtype="category",
            nullable=True,
            checks=[*category_validation(("<50", "50-59", "60-69", "70-79", "80+"), ordered=True)]
            ),
        'high_grade': pa.Column(
            dtype="boolean",
            nullable=False
            ),
        'multiple_meningiomas': pa.Column(
            dtype="boolean",
            nullable=False
            ),
        'ki67_mid': pa.Column(
            dtype="Float64",
            nullable=True
            ),
        'ki67_group': pa.Column(
            dtype="category",
            nullable=True,
            checks=[*category_validation(("low_le_4", "intermediate_5_9", "high_ge_10"), ordered=True)]
            ),
    
    },
    strict=True
)
schema_validation(df, lazy=True).head()

,id,patient_code,entry_year,age,sex,who_grade,progesterone_pos,ki67_pct,brain_invasion,hist_necrosis,mri_date,side,tumor_location,meningioma_count,max_diameter_cm,tumor_volume,additional_ct,iv_contrast,tumor_episode,tumor_margin,dural_tail,capsular_enhancement,heterogeneous_enhancement,perifocal_edema,edema_volume_cm3,mass_effect,calcification,cystic_component,mri_necrosis,hemorrhage,hyperostosis,cortical_destruction,dwi_hyperintensity,t2_hyperintensity,t1_hypointensity,sinus_invasion,transfalcine_extension,adc_value,age_bins,high_grade,multiple_meningiomas,ki67_mid,ki67_group
1,2.0,070458-11352,2025,67.0,female,1,True,1-3,False,False,2025-06-27,right,skull_base,2,4.9,36.5,False,True,primary,regular,False,True,False,True,5.0,True,True,False,False,False,False,False,True,True,True,no_invasion,False,0.88,60-69,False,True,2.0,low_le_4
2,3.0,230949-11093,2025,76.0,female,1,True,1-5,False,False,2025-09-05,midline,skull_base,1,2.8,6.86,True,True,primary,irregular,False,True,False,True,26.0,True,False,False,False,False,False,False,True,True,True,no_invasion,True,0.94,70-79,False,False,3.0,low_le_4
3,4.0,140352-11498,2025,73.0,female,2,True,25-30,True,True,2025-07-23,right,non_skull_base,1,4.7,40.9,True,True,recurrent,irregular,False,True,True,True,135.0,True,True,True,False,True,False,False,True,True,True,no_invasion,False,0.6,70-79,True,False,27.5,high_ge_10
4,5.0,151269-12200,2025,55.0,male,1,True,1-2,False,False,2025-08-04,right,non_skull_base,1,3.7,8.3,True,True,primary,irregular,True,True,False,True,24.0,True,True,False,False,False,False,False,True,True,True,no_invasion,False,1.2,50-59,False,False,1.5,low_le_4
5,6.0,270866-10213,2025,58.0,female,1,True,1-2,False,False,2025-12-27,right,non_skull_base,1,2.9,4.43,False,True,primary,irregular,True,True,False,False,0.0,True,False,False,False,False,True,False,True,True,True,no_invasion,False,0.93,50-59,False,False,1.5,low_le_4


## 14 · Pre-imputation: poke it with a stick

🔬 Free-form scratch space — last look before the gaps get filled.

<details>
<summary>🔧 How it works</summary>

- 👀 Eyeball the cleaned, validated cohort: distributions, cross-tabs, suspicious values.
- 🧪 Nothing here is part of the pipeline.
- 🛑 A final sanity check before §15 imputes.

</details>

<details>
<summary>💡 Code examples &amp; scenarios</summary>

**Quick distribution / cross-tab checks**
```python
df["who_grade"].value_counts(dropna=False)
df.groupby("high_grade")["age"].describe()
pd.crosstab(df["sex"], df["high_grade"], dropna=False)
```

</details>

In [26]:
counts = pd.crosstab(df["additional_ct"], df["hyperostosis"])

percentages = (
    pd.crosstab(
        df["additional_ct"],
        df["hyperostosis"],
        normalize="index"
    )
    .mul(100)
    .round(1)
)

counts.astype(str) + " (" + percentages.astype(str) + "%)"

# CT detected extra hyperostosis missed on MRI, or
# suspected bone involvement prompted clinicians to order CT???
# So hyperostosis was recorded about 50% more often when additional CT was available???

hyperostosis,False,True
additional_ct,,
False,211 (77.6%),61 (22.4%)
True,52 (65.8%),27 (34.2%)


## 15 · Imputation (MICE or simple)

🧬 Fill the gaps — generate **m** completed datasets for Rubin's-rules pooling downstream.

<details>
<summary>🔧 How it works · <code>missingness_resolution</code></summary>

Three engines, pick one:

- 🥇 **`proper_mice_impute`** (primary) — formal mixed-type MICE via R's `mice` (needs R + `mice`/`jsonlite`). Drops non-outcome derived columns, imputes their sources, recreates them via `apply_post_mice_derivations`, and saves diagnostics under `output/missingness/mice/`.
- 🧪 **`rf_chained_impute`** (optional) — RandomForest chained imputation; sensitivity check only, **not** valid for Rubin pooling.
- 🩹 **`simple_impute_stage`** (fallback) — median/mode, single dataset.

Tips:

- 🔢 `m≥10` (e.g. 20) for publication; `m=3` for a quick smoke run.
- 💾 Writes `unimputed_df.parquet` + `mice_imputed_df.parquet`, then pandera-validates every draw.

</details>

<details>
<summary>💡 Code examples &amp; scenarios</summary>

**Primary — formal MICE (R)**
```python
imputed_frames = proper_mice_impute(
    df, schema,
    m=3, max_iter=5,             # 🔁 3/5 smoke · 20/20 publication
    random_state=42,
    analysis_outcome="high_grade",
    derived_dependencies=DERIVED_DEPENDENCIES,
    post_impute_transform=apply_post_mice_derivations,
    output_root=OUTPUT_ROOT,
)
for _frame in imputed_frames:
    schema_validation(_frame, lazy=True)
```

**Optional — RF chained (sensitivity only)**
```python
imputed_frames_rf = rf_chained_impute(
    df, schema, m=3, max_iter=10, n_estimators=20, output_root=OUTPUT_ROOT,
)
```

**Fallback — simple median/mode**
```python
imputed_frames = [simple_impute_stage(df, schema, OUTPUT_ROOT, impute_binary=False)]
```

</details>


In [27]:
#🟧🟧🟧 Formal mixed-type MICE (R mice) — primary imputation (see §15 notes)
# Requires R + packages: install.packages(c("mice", "jsonlite"))

imputed_frames = proper_mice_impute(
    df,
    schema,
    m=20,                     # 3 smoke | 20 publication
    max_iter=20,              # 5 smoke | 20 publication
    random_state=42,
    analysis_outcome="high_grade",
    derived_dependencies=DERIVED_DEPENDENCIES,
    post_impute_transform=apply_post_mice_derivations,
    output_root=OUTPUT_ROOT,
)

# Pandera validation on EVERY completed dataset (not only draw 1).
for _i, _frame in enumerate(imputed_frames, start=1):
    schema_validation(_frame, lazy=True)
    print(f"✅ Pandera validated imputed frame {_i}/{len(imputed_frames)}")

💾 Saved unimputed cohort → output\datasets\unimputed_df.parquet
🧬 Formal mixed-type MICE (R mice) — m=20, maxit=20, 10 incomplete variables…
   📥 Loaded input: 352 rows x 35 cols
   🔧 Types set for 34 declared columns
   🧮 Predictor matrix built (35 predictors)
   🧬 Running MICE: 20 draws x 20 iterations…
   ✅ MICE converged (no method overrides)
   💾 Wrote 20 completed dataset(s)
   📈 Chain diagnostics saved
   🏁 run_mice.R OK | m=20 maxit=20 incomplete_vars=10 logged_events=0
💾 Saved 20 imputed frames → output\missingness\mice
💾 Saved mice modelling cohort → output\datasets\mice_imputed_df.parquet
🏁 Formal MICE complete — 20 completed datasets, diagnostics in output\missingness\mice
✅ Pandera validated imputed frame 1/20
✅ Pandera validated imputed frame 2/20
✅ Pandera validated imputed frame 3/20
✅ Pandera validated imputed frame 4/20
✅ Pandera validated imputed frame 5/20
✅ Pandera validated imputed frame 6/20
✅ Pandera validated imputed frame 7/20
✅ Pandera validated imputed frame

In [28]:
#🟧🟧🟧 OPTIONAL sensitivity analysis — RF chained imputation (NOT formal MICE)
# Post-hoc Bernoulli sampling; Rubin pooling is NOT supported on these draws.
# Manifest marks proper_multiple_imputation=False. Use only as a sensitivity check.

#imputed_frames_rf = rf_chained_impute(
#    df, schema, m=3, max_iter=10, n_estimators=20, output_root=OUTPUT_ROOT,
#)


#🟧🟧🟧 Skip MICE — median/mode imputation (binary left NaN by default)

#imputed_frames = [simple_impute_stage(df, schema, OUTPUT_ROOT, impute_binary=False)]

#display(imputation_audit(
#    load_unimputed_dataset(OUTPUT_ROOT),
#    load_modeling_frames(OUTPUT_ROOT)[0],
#    schema,
#    INFERENTIAL_MANUAL_PREDICTORS,
#    impute_binary=False,
#))

#print("NaN count (all columns):", load_modeling_frames(OUTPUT_ROOT)[0].isna().sum().sum())

## 16 · Save handoff datasets

📦 Reload and validate the parquets so the modelling notebook can trust them.

<details>
<summary>🔧 How it works · <code>dataset_handoff</code></summary>

- 💾 Parquets are written during imputation (§15); this step doesn't re-impute.
- 🔁 `validate_handoff_datasets` reloads every parquet and round-trip validates it (schema + row counts).
- 🤝 Guarantees `meningioma-modelling.ipynb` reads exactly what was saved.
- 🏷️ Set `imputation_method` to match what you ran (`"mice"` vs `"simple"`).

</details>

<details>
<summary>💡 Code examples &amp; scenarios</summary>

**MICE handoff (multiple draws)**
```python
imputation_method = "mice"
validate_handoff_datasets(
    OUTPUT_ROOT, df_unimputed=df,
    imputation_method="mice", imputed_frames=imputed_frames,
)
```

**Simple handoff (single frame)**
```python
imputation_method = "simple"
validate_handoff_datasets(
    OUTPUT_ROOT, df_unimputed=df,
    imputation_method="simple", imputed_single=imputed_frames[0],
)
```

</details>


In [29]:
# Set imputation_method to "simple" if you used simple_impute_stage above instead of MICE.
imputation_method = "mice"

if imputation_method == "mice":
    validate_handoff_datasets(
        OUTPUT_ROOT,
        df_unimputed=df,
        imputation_method="mice",
        imputed_frames=imputed_frames,
    )
else:
    imputed_single = imputed_frames[0]
    validate_handoff_datasets(
        OUTPUT_ROOT,
        df_unimputed=df,
        imputation_method="simple",
        imputed_single=imputed_single,
    )


✅ parquet roundtrip validated — unimputed_df.parquet
✅ parquet roundtrip validated — mice_imputed_df.parquet
✅ parquet roundtrip validated — imputed_001.parquet
✅ parquet roundtrip validated — imputed_002.parquet
✅ parquet roundtrip validated — imputed_003.parquet
✅ parquet roundtrip validated — imputed_004.parquet
✅ parquet roundtrip validated — imputed_005.parquet
✅ parquet roundtrip validated — imputed_006.parquet
✅ parquet roundtrip validated — imputed_007.parquet
✅ parquet roundtrip validated — imputed_008.parquet
✅ parquet roundtrip validated — imputed_009.parquet
✅ parquet roundtrip validated — imputed_010.parquet
✅ parquet roundtrip validated — imputed_011.parquet
✅ parquet roundtrip validated — imputed_012.parquet
✅ parquet roundtrip validated — imputed_013.parquet
✅ parquet roundtrip validated — imputed_014.parquet
✅ parquet roundtrip validated — imputed_015.parquet
✅ parquet roundtrip validated — imputed_016.parquet
✅ parquet roundtrip validated — imputed_017.parquet
✅ parqu